![image_1778495334718.png](./image_1778495334718.png "image_1778495334718.png")

# Spark Structured Streaming Aggregations & State Store — Professor Style Notes

Welcome back.

In this lecture, we are going to understand one of the most important concepts in Spark Structured Streaming.

We will learn:

1. How Spark calculates aggregates across micro-batches
2. What is a State Store
3. Why State Store is required
4. How append mode works
5. How complete mode works
6. Why complete mode can become a performance bottleneck

And to understand all these concepts properly, we are going to build a practical streaming application.

So let’s start carefully and step by step.

---

# 1. Problem Statement

Suppose we already built a streaming pipeline earlier.

We already have:

* A Bronze Layer Stream
* A Silver or Gold Layer Stream

Now we want to extend that pipeline.

We will use the same invoice dataset again.

Our requirement is very simple.

---

## Requirement 1 — Bronze Layer

Read invoice JSON files from the Landing Zone and store them into a Bronze Delta Table.

This is a raw ingestion process.

No transformations.

No business logic.

Just capture raw data.

---

## Requirement 2 — Gold Layer

Now we want another streaming job.

This job will:

* Read data continuously from the Bronze table
* Calculate customer-wise aggregates

Specifically:

* Total Sales per Customer
* Reward Points per Customer

And reward points are simply:

* 2% of Total Sales

So mathematically:

Reward Points = Total Sales × 0.02

Very simple requirement.

But behind this simple requirement lies one extremely powerful concept:

# Stateful Stream Processing

And that is what we are going to learn.

---

# 2. Understanding the Main Challenge

Before writing code, think carefully.

Spark Structured Streaming runs using:

# Micro-Batches

That means:

* First micro-batch processes some records
* Second micro-batch processes new incoming records
* Third micro-batch processes more incoming records

Now the question is:

# How does Spark calculate TOTAL aggregates across all micro-batches?

Because each micro-batch only sees NEW DATA.

It does not re-read old data every time.

This is the key concept.

---

# 3. Bronze Layer Streaming Job

Let us first build the Bronze Layer.

Its responsibility is simple:

* Read JSON files from Landing Zone
* Store them into a Delta table

---

# 4. Reading Streaming Data

We use:

```python
spark.readStream
```

Format:

```python
.format("json")
```

And we explicitly provide schema.

This is very important.

In production streaming pipelines, always provide schema explicitly.

Schema inference in streaming is not recommended.

---

# 5. Adding Input File Name

After reading the data, we add one extra column:

```python
input_file_name()
```

This is a very good practice.

Why?

Because later during debugging or troubleshooting, we can identify:

* Which record came from which file

This becomes extremely useful in real-world systems.

Especially in:

* Data lakes
* Bronze ingestion layers
* Audit pipelines

---

# 6. Writing to Bronze Table

Now we write the stream.

We use:

```python
writeStream
```

And then:

```python
.toTable("invoices")
```

This creates or writes into a Delta table.

---

# 7. Append Output Mode

Now comes our first important concept.

We use:

# Append Mode

```python
.outputMode("append")
```

What does append mode mean?

It means:

# “Whatever records are produced in the current micro-batch, append them into the target.”

Simple.

---

## Example

Suppose:

### Micro-Batch 1

Reads:

* 5000 records

Spark appends:

* 5000 rows into table

---

### Micro-Batch 2

Reads:

* 6000 records

Spark appends:

* Another 6000 rows

This is:

# Insert-Only Streaming

No updates.

No overwrites.

No merges.

Just continuously appending new rows.

That is why append mode is perfect for Bronze ingestion.

---

# 8. Gold Layer Streaming Job

Now comes the interesting part.

We read continuously from Bronze table.

```python
spark.readStream.table("invoices")
```

This creates a streaming DataFrame from Delta table.

---

# 9. Calculating Aggregates

Now we apply:

```python
groupBy(customer_card_no)
```

And calculate:

```python
sum(total_amount)
```

And reward points:

```python
sum(total_amount) * 0.02
```

So logically we are calculating:

# Customer-wise running totals

This is stateful aggregation.

---

# 10. The Big Problem

Now think carefully.

Suppose:

## First Micro-Batch

Customer 101 purchases:

₹36,859

Spark calculates:

```text
Customer 101 → 36,859
```

Good.

Now second micro-batch arrives.

Customer 101 again purchases:

₹20,740

Now what should be final result?

Not:

```text
20,740
```

Correct result should be:

```text
36,859 + 20,740 = 57,599
```

But second micro-batch only reads NEW RECORDS.

So how does Spark remember:

```text
36,859
```

from previous batch?

This is where:

# State Store

comes into picture.

---

# 11. What is State Store?

State Store is one of the most important concepts in Structured Streaming.

Think of it like this:

# State Store = Spark’s Memory Across Micro-Batches

Spark remembers previously calculated aggregates inside the State Store.

---

# 12. Where is State Store Stored?

State Store is stored inside:

# Checkpoint Directory

We already know checkpoints store:

* Processed file metadata
* Offsets
* Recovery information

But additionally:

# Aggregation states are also stored there

Especially for:

* Aggregations
* Stream-stream joins
* Stateful operations

---

# 13. How State Store Works

Let us understand step-by-step.

---

# First Micro-Batch

Suppose first batch processes:

9 invoices for Customer 101

Total becomes:

```text
36,859
```

Spark:

1. Writes result to output table
2. Stores state internally

State Store now contains:

```text
Customer 101 → 36,859
```

---

# Second Micro-Batch

Now second batch processes:

4 new invoices

Their sum is:

```text
20,740
```

Spark now:

1. Reads previous state from State Store
2. Finds:

```text
36,859
```

3. Adds current batch total:

```text
20,740
```

4. Produces:

```text
57,599
```

5. Updates State Store

Now State Store becomes:

```text
Customer 101 → 57,599
```

---

# Third Micro-Batch

Suppose third batch total is:

```text
31,959
```

Spark reads previous state:

```text
57,599
```

Adds current:

```text
31,959
```

Final:

```text
89,558
```

And updates State Store again.

---

# 14. Why State Store is Powerful

Without State Store:

Every micro-batch would need to:

* Re-read all historical data
* Recalculate everything from scratch

That would be extremely slow.

Instead Spark only processes:

# Incremental Data

and combines it with stored state.

This is why Structured Streaming scales beautifully.

---

# 15. Complete Output Mode

Now comes another very important concept.

We used:

```python
.outputMode("complete")
```

What does complete mode mean?

It means:

# “After every micro-batch, output the COMPLETE latest result.”

Not just current batch output.

Entire latest aggregate result.

---

## Example

Suppose:

State Store contains:

```text
Customer A → 500
Customer B → 700
Customer C → 900
```

After every micro-batch, Spark outputs ALL THREE rows again.

Even if only Customer A changed.

That is complete mode.

---

# 16. Why Complete Mode is Useful

Complete mode is useful when:

* Result set is small
* Overwriting target table is acceptable
* Simplicity is more important than efficiency

Example:

* Small dashboards
* Small aggregates
* Demo pipelines
* POCs
* Learning projects

---

# 17. Why Complete Mode is Expensive

Now think about scale.

Suppose:

* 20 million customers
* Every micro-batch recalculates all customer totals
* Entire table rewritten every time

That becomes extremely expensive.

Because complete mode repeatedly outputs:

# Entire State Store

every micro-batch.

This causes:

* Heavy I/O
* Large table overwrites
* Slow streaming performance

---

# 18. Append Mode vs Complete Mode

| Feature           | Append Mode         | Complete Mode        |
| ----------------- | ------------------- | -------------------- |
| Output Type       | Only new rows       | Entire result        |
| Best For          | Insert-only streams | Aggregations         |
| Overwrites Table? | No                  | Yes                  |
| Performance       | Faster              | Can become expensive |
| Uses State Store? | Usually no          | Yes                  |

---

# 19. Real Production Challenge

In real production systems:

* Millions of keys
* Huge aggregates
* Massive state size

So complete mode becomes inefficient.

Then how do we solve this?

Instead of rewriting everything:

# We update only changed rows

And that is done using:

# Update Mode + Merge Operations

Which we will learn in the next lecture.

---

# 20. Final Understanding

Let us summarize everything carefully.

---

# Important Concepts Learned

## 1. Structured Streaming runs using Micro-Batches

Each micro-batch processes only new incoming data.

---

## 2. Aggregations require memory across batches

Spark must remember previous totals.

---

## 3. State Store solves this problem

State Store stores aggregation state across micro-batches.

---

## 4. State Store lives inside Checkpoint Directory

Checkpoint is not only for fault tolerance.

It also stores streaming state.

---

## 5. Append Mode

Outputs only new rows from current micro-batch.

Best for insert-only workloads.

---

## 6. Complete Mode

Outputs entire aggregate result every micro-batch.

Useful but can become expensive at large scale.

---

# Key Mental Model

Think like this:

```text
Micro-Batch = New Data
State Store = Memory
Complete Mode = Full Snapshot Output
Append Mode = Incremental Output
```

If you remember this mental model, the entire aggregation architecture of Spark Structured Streaming becomes very easy to understand.

---

That’s all for this lecture.

See you again in the next lecture.

Keep learning and keep growing.


# Streaming Aggregates and Incremental Aggregation in Spark Structured Streaming

Welcome back.

In the previous lectures, we learned how Spark Structured Streaming works using micro-batches, how streams can be chained together, and how data flows through Bronze, Silver, and Gold layers.

Now in these lectures, we are moving into one of the most important real-world concepts in streaming systems:

* Real-time aggregations
* Stateful stream processing
* State Store
* Output modes
* Incremental aggregations
* Update mode
* foreachBatch + MERGE pattern

These concepts are extremely important because almost every real-world streaming pipeline performs aggregations.

Examples:

* Total sales per customer
* Total clicks per user
* Running balance
* Fraud detection counters
* Reward points
* Session statistics
* Real-time dashboards

So let us understand everything carefully.

---

# Part 1 — Understanding Streaming Aggregations

---

# The Requirement

We want to build a simple streaming pipeline.

We already have:

* Landing Zone
* Bronze Layer

Now we want to build:

* Gold Layer with aggregations

Our requirement is:

---

## Step 1 — Bronze Layer

Read invoice JSON files from the Landing Zone and store them as raw data into a Bronze Delta table.

No transformation.

Just ingestion.

---

## Step 2 — Gold Layer

Read data from the Bronze table in streaming mode and calculate:

* Total Sales per Customer
* Reward Points per Customer

Reward points are:

* 2% of total sales

Very simple aggregation.

---

# Architecture

The complete pipeline looks like this:

```text
Landing Zone
      ↓
Bronze Stream
      ↓
Bronze Delta Table
      ↓
Gold Stream
      ↓
Customer Rewards Table
```

---

# Important Concepts We Want to Learn

This example helps us understand three major concepts:

1. How Spark calculates aggregates across micro-batches
2. What is State Store
3. Output Modes

   * Append Mode
   * Complete Mode
   * Update Mode

---

# Bronze Layer Streaming Job

The Bronze Layer job is simple.

It performs:

```text
Landing Zone → Bronze Delta Table
```

---

# Reading Streaming Data

We use:

```python
spark.readStream
```

Format:

```python
.format("json")
```

Schema:

```python
.schema(invoiceSchema)
```

Load:

```python
.load(landingZone)
```

---

# Best Practice — Add Input File Name

While ingesting raw files, we should always capture:

```python
input_file_name()
```

Why?

Because later:

* debugging becomes easier
* traceability improves
* lineage becomes easier

Each record knows from which file it came.

This is a very good Bronze Layer practice.

---

# Writing to Bronze Table

Now we write the stream into a Delta table.

We use:

```python
.writeStream
```

Important options:

```python
.queryName()
.option("checkpointLocation")
.outputMode("append")
.toTable()
```

---

# Understanding Append Mode

This is the first output mode.

In append mode:

```text
Each micro-batch appends new rows only.
```

Example:

---

## Micro-batch 1

Reads:

```text
5000 rows
```

Writes:

```text
5000 inserts
```

---

## Micro-batch 2

Reads:

```text
6000 rows
```

Writes:

```text
6000 inserts
```

---

Nothing is updated.

Nothing is overwritten.

Only INSERTS happen.

That is why append mode is perfect for:

* Bronze Layer
* Raw ingestion
* Insert-only systems

---

# Gold Layer Streaming Job

Now comes the interesting part.

We read from the Bronze table:

```python
spark.readStream.table("invoices")
```

---

# Aggregation Logic

We group data by customer:

```python
groupBy("customer_card_no")
```

Then calculate:

* total sales
* total reward points

Using:

```python
sum("total_amount")
```

And:

```python
expr("sum(total_amount) * 0.02")
```

---

# The Big Problem

Now think carefully.

Streaming runs in micro-batches.

Suppose:

---

## First Micro-batch

Customer 101 purchases:

```text
₹10,000
```

---

## Second Micro-batch

Same customer purchases again:

```text
₹5,000
```

---

What should final total be?

```text
₹15,000
```

Correct.

But here is the challenge:

Second micro-batch reads only NEW data.

It does NOT reread old files.

So how does Spark remember previous totals?

This is where State Store comes into the picture.

---

# Understanding State Store

State Store is one of the most important concepts in Spark Structured Streaming.

Whenever Spark performs:

* aggregations
* joins
* windowing
* deduplication
* stateful operations

Spark must remember previous results.

That memory is called:

# State

And the storage area that keeps this state is called:

# State Store

---

# Where is State Store Stored?

By default:

```text
Inside the checkpoint directory
```

So checkpoint directory stores:

* offsets
* metadata
* processed files
* aggregation state
* join state

Everything needed for fault tolerance.

---

# How State Store Works

Let us understand step-by-step.

---

# Example Data

Suppose Customer 101 has:

---

## File 1

Invoices total:

```text
₹36,859
```

---

## File 2

Invoices total:

```text
₹20,740
```

---

## File 3

Invoices total:

```text
₹31,959
```

---

# Micro-batch 1

Spark reads File 1.

Calculates:

```text
₹36,859
```

Then:

* writes result to target table
* stores this value in State Store

State Store now contains:

```text
Customer 101 → ₹36,859
```

---

# Micro-batch 2

Spark reads ONLY File 2.

Calculates:

```text
₹20,740
```

Then Spark checks State Store.

Finds previous total:

```text
₹36,859
```

Then combines:

```text
36,859 + 20,740
= 57,599
```

Now State Store updates itself:

```text
Customer 101 → ₹57,599
```

---

# Micro-batch 3

Spark reads ONLY File 3.

Calculates:

```text
₹31,959
```

Reads previous state:

```text
₹57,599
```

Combines:

```text
57,599 + 31,959
= 89,558
```

Updates State Store again.

---

# This is Extremely Powerful

Notice something important.

Spark:

* never rereads old data
* never recalculates entire history
* still produces cumulative totals

Why?

Because State Store remembers previous state.

This is the foundation of stateful stream processing.

---

# Complete Output Mode

Now let us understand the second output mode.

We used:

```python
.outputMode("complete")
```

---

# What Complete Mode Does

Complete mode means:

```text
After every micro-batch,
send the ENTIRE result table.
```

Not only changed rows.

Everything.

---

# Example

Suppose:

```text
10,000 customers
```

Even if only:

```text
5 customers changed
```

Complete mode still outputs:

```text
all 10,000 rows
```

every micro-batch.

---

# Why Complete Mode Exists

Because aggregations require state.

Spark already has full state in State Store.

So it can reconstruct the entire result set.

---

# Problem with Complete Mode

It is simple.

But not scalable.

Imagine:

```text
20 million customers
```

Rewriting entire table every micro-batch becomes extremely expensive.

Problems:

* high I/O
* expensive overwrites
* slow performance
* poor scalability

So Complete Mode works only for:

* small result sets
* POCs
* learning
* prototypes

Not ideal for large production systems.

---

# Part 2 — Incremental Aggregation

Now we move to the advanced approach.

Instead of rewriting everything:

We want incremental updates only.

This is the correct production approach.

---

# The Goal

We want:

```text
Only NEW or CHANGED aggregates
```

to be written.

Not entire table.

---

# Update Mode

This introduces the third output mode:

# Update Mode

```python
.outputMode("update")
```

---

# What Update Mode Does

Update mode outputs only:

* new rows
* modified rows

Nothing else.

---

# Example

Suppose:

```text
10,000 customers exist
```

Only:

```text
5 customers changed
```

Then update mode outputs only:

```text
5 rows
```

This is extremely efficient.

---

# But There is a Problem

Spark's:

```python
toTable()
```

does not support MERGE.

It only supports:

* INSERT
* OVERWRITE

But we need UPSERT behavior.

Meaning:

* update existing customers
* insert new customers

So we need:

# MERGE INTO

---

# Solution — foreachBatch

This is the most important production pattern.

We use:

```python
.foreachBatch()
```

Inside foreachBatch:

* Spark gives us micro-batch DataFrame
* We write custom logic
* We execute MERGE

This is one of the most widely used enterprise streaming patterns.

---

# Flow of Incremental Aggregation

The flow becomes:

```text
Streaming Aggregation
        ↓
Update Mode
        ↓
Only changed rows
        ↓
foreachBatch()
        ↓
MERGE INTO target table
```

---

# Upsert Logic

Inside foreachBatch:

---

## When Matched

Update totals.

```sql
WHEN MATCHED THEN UPDATE
```

---

## When Not Matched

Insert new customer.

```sql
WHEN NOT MATCHED THEN INSERT
```

---

# Why This is Better

This approach is highly scalable because:

* only changed rows move
* only changed rows merge
* no full-table rewrite
* minimal I/O

This is the preferred production architecture.

---

# RocksDB State Store Provider

Now another important optimization.

State Store operations involve:

* reads
* writes
* serialization
* disk I/O

Default state store provider is slower.

Spark introduced:

# RocksDB State Store Provider

Much faster for large stateful workloads.

---

# Enabling RocksDB

We set:

```python
spark.conf.set(
  "spark.sql.streaming.stateStore.providerClass",
  "org.apache.spark.sql.execution.streaming.state.RocksDBStateStoreProvider"
)
```

---

# Why RocksDB Helps

It improves:

* state read speed
* state write speed
* memory efficiency
* large-state scalability

Very useful when:

* millions of keys exist
* aggregations are huge
* state becomes large

---

# Summary of Output Modes

| Output Mode | Behavior                       | Best Use Case           |
| ----------- | ------------------------------ | ----------------------- |
| Append      | Writes only new rows           | Bronze/raw ingestion    |
| Complete    | Writes full result every batch | Small aggregations      |
| Update      | Writes only changed rows       | Production aggregations |

---

# Summary of Stateful Aggregation

---

## Spark Structured Streaming

Processes data incrementally using micro-batches.

---

## Aggregations Need Memory

Spark must remember previous totals.

That memory is stored in:

# State Store

---

## State Store

Stored inside:

```text
Checkpoint Directory
```

---

## Complete Mode

Outputs:

```text
Entire result set
```

every micro-batch.

Simple but inefficient.

---

## Update Mode

Outputs:

```text
Only changed aggregates
```

Much more scalable.

---

## foreachBatch + MERGE

This is the enterprise production approach.

Used everywhere in real-time systems.

---

# Final Mental Model

Think about it like this:

---

# Append Mode

```text
Keep adding rows
```

---

# Complete Mode

```text
Rewrite everything
```

---

# Update Mode

```text
Modify only what changed
```

---

# Final Takeaway

If you truly understand:

* State Store
* Output Modes
* foreachBatch
* Incremental Aggregation
* MERGE pattern

then you understand the core foundation of real-world Spark Structured Streaming systems.

These concepts are used in:

* Lakehouse pipelines
* Real-time analytics
* Fraud systems
* Recommendation engines
* Financial systems
* Reward systems
* Streaming ETL
* CDC pipelines

This is one of the most important topics in Spark Structured Streaming.

See you again in the next lecture.

Keep learning and keep growing.


![image_1778498957998.png](./image_1778498957998.png "image_1778498957998.png")
![image_1778498973076.png](./image_1778498973076.png "image_1778498973076.png")
![image_1778498911999.png](./image_1778498911999.png "image_1778498911999.png")

# Monitoring Spark Structured Streaming Queries and Understanding State Store Memory

Welcome back.

So far, we have learned many important concepts about Spark Structured Streaming.

We learned:

* How Spark Structured Streaming works internally
* Micro-batch execution model
* `readStream`
* `writeStream`
* Output modes

  * Append
  * Complete
  * Update
* Stateful aggregations
* Incremental aggregations
* `foreachBatch`
* MERGE pattern
* State Store

Now, in this lecture, we are going to learn another very important production-level topic:

# Monitoring Streaming Queries

Because in real-world projects, writing streaming code is only half the work.

The other half is:

* Monitoring
* Debugging
* Performance tuning
* Understanding bottlenecks
* Investigating memory issues
* Monitoring State Store growth

And for all these things, the most important place is:

# Spark UI

This lecture is all about understanding:

* How to monitor Spark Structured Streaming queries
* How to inspect micro-batches
* How to inspect State Store statistics
* How to monitor memory usage
* How to analyze streaming performance

This is extremely important for production systems.

---

# Recap — What Happens Behind the Scene?

Before moving ahead, let us quickly revise something important.

Whenever we create a streaming query like this:

```python
spark.readStream ...
    .writeStream ...
```

Spark creates:

# A Background Streaming Query Thread

This thread continuously runs in the background.

Its job is:

```text
Trigger Micro-batches
        ↓
Read new data
        ↓
Process data
        ↓
Write results
        ↓
Repeat forever
```

So every streaming query is actually a continuously running background process.

---

# One Query = One Background Thread

Suppose you create:

```python
Bronze Stream
Gold Stream
Silver Stream
```

Then Spark creates:

```text
3 separate streaming query threads
```

Each query has:

* its own checkpoints
* its own state
* its own micro-batches
* its own execution statistics

And all these can be monitored from Spark UI.

---

# Where to Monitor Streaming Queries?

Go to:

# Spark UI

This is the most important monitoring tool in Spark.

If you are serious about Spark engineering, you must become very comfortable with Spark UI.

---

# Structured Streaming Tab

Inside Spark UI, there is a tab called:

# Structured Streaming

This tab shows all active and completed streaming queries.

---

# Example Queries

In our example, we created two queries:

---

## Bronze Query

Query name:

```python
"bronze_ingestion"
```

This reads from Landing Zone and writes into Bronze table.

---

## Gold Query

Query name:

```python
"gold_update"
```

This reads from Bronze table and calculates incremental aggregates.

---

# What We See in Spark UI

Inside Structured Streaming tab, Spark shows:

* Query Name
* Query Status
* Run ID
* Start Time
* Batch Information
* Input Rate
* Processing Rate
* Batch Duration

and much more.

---

# Query Status

You may see:

```text
RUNNING
```

or:

```text
FINISHED
```

If the stream is active:

```text
RUNNING
```

If execution completed:

```text
FINISHED
```

Even after completion, Spark UI still preserves execution statistics.

This is very useful for debugging.

---

# Understanding Batch Numbers

Suppose Spark UI shows:

```text
Latest Batch = 1
```

What does it mean?

Remember:

Micro-batches start from:

```text
0
```

So:

```text
Batch 0
Batch 1
```

means:

# Two micro-batches already executed

This is important while debugging streaming jobs.

---

# Important Streaming Metrics

Now let us understand the most important metrics available in Spark UI.

---

# 1. Input Rate

This shows:

# How many records per second are arriving

Example:

```text
5000 rows/sec
```

This tells us how fast data is entering the streaming pipeline.

---

# 2. Processing Rate

This shows:

# How many records per second Spark is processing

This is extremely important.

Because if:

```text
Input Rate > Processing Rate
```

then your stream will slowly lag behind.

This is called:

# Backpressure

and eventually your system becomes slower and slower.

---

# Example

Suppose:

| Metric          | Value           |
| --------------- | --------------- |
| Input Rate      | 10,000 rows/sec |
| Processing Rate | 5,000 rows/sec  |

Problem:

Spark cannot keep up with incoming data.

Data backlog starts increasing.

Very dangerous in production.

---

# 3. Input Rows

Shows:

# Number of rows processed in each micro-batch

Useful for understanding:

* batch sizes
* ingestion volume
* traffic spikes

---

# 4. Batch Duration

Shows:

# How much time each micro-batch took

This is one of the most important metrics.

Because streaming systems are expected to process data quickly.

If batch duration increases continuously:

```text
System is struggling
```

Possible reasons:

* large state store
* skewed data
* insufficient memory
* expensive transformations
* joins
* slow merge operations

---

# Stateful Streaming Metrics

Now comes the most important section.

Our Gold query performs:

# Aggregations

Aggregations are:

# Stateful Operations

Which means:

Spark must maintain State Store.

Because of that, Spark UI shows additional State Store metrics.

---

# State Store Statistics

Inside Structured Streaming UI, Spark shows special metrics related to State Store.

These metrics are extremely important in production systems.

---

# 1. Total State Store Rows

This shows:

# How many keys currently exist in State Store

Example:

```text
100 rows
```

Meaning:

Spark is maintaining aggregate state for:

```text
100 unique customers
```

Not full invoices.

Only aggregate state.

Example:

```text
Customer ID
Total Sales
Reward Points
```

That is what State Store stores.

---

# Example Scenario

---

## First Micro-batch

Suppose:

```text
65 unique customers
```

Then State Store contains:

```text
65 rows
```

---

## Second Micro-batch

Suppose:

* some existing customers purchased again
* some new customers appeared

Now State Store grows to:

```text
100 rows
```

This means:

Spark is now maintaining state for:

```text
100 unique customers
```

---

# 2. Updated State Store Rows

This metric shows:

# How many state records changed in this micro-batch

Very useful metric.

---

# Example

Suppose:

```text
100 total customers
```

But:

```text
98 customers purchased again
```

Then:

```text
98 state rows updated
```

This helps us understand:

* how active the stream is
* how many aggregates changed
* how much state churn exists

---

# 3. State Store Memory Usage

This is one of the MOST IMPORTANT metrics.

Spark UI shows:

# Memory used by State Store

Example:

```text
128 MB
170 MB
180 MB
```

This is extremely important.

Because Stateful Streaming systems are memory-heavy systems.

---

# Why Memory Keeps Growing

Remember:

State Store keeps aggregation state.

As more customers arrive:

* more keys appear
* more aggregate data is stored
* more memory is consumed

So naturally:

```text
State Store size grows
```

---

# Example Growth

| Micro-batch | Memory Usage |
| ----------- | ------------ |
| Batch 0     | 128 MB       |
| Batch 1     | 170 MB       |
| Batch 2     | 180 MB       |

This tells us:

State Store is continuously growing.

---

# Why This Matters

Because large State Store creates problems:

* high memory usage
* expensive serialization
* slower processing
* longer micro-batches
* higher GC pressure
* out-of-memory risk

This is one of the biggest challenges in stream processing systems.

---

# Extremely Important Insight

Batch processing problems are mostly:

```text
CPU problems
```

Streaming problems are often:

```text
STATE problems
```

This is a very important distinction.

Managing State Store efficiently is one of the most critical skills in real-time data engineering.

---

# What Causes Large State Stores?

Many operations create state:

* aggregations
* streaming joins
* windowing
* deduplication
* sessionization

All these operations store intermediate state.

---

# Real-World Example

Suppose you have:

```text
20 million customers
```

and you are maintaining:

```text
running totals
```

for all customers.

State Store becomes huge.

Memory usage explodes.

Streaming performance drops.

This is why State management is a major topic in Spark Structured Streaming.

---

# Monitoring Streaming Systems

Whenever you work with Stateful Streaming:

Always monitor:

* State Store rows
* State Store memory
* Batch duration
* Processing rate
* Input rate

These metrics tell you whether your system is healthy.

---

# Important Mental Model

Think of State Store as:

# A continuously growing in-memory aggregation table

Spark keeps updating this table after every micro-batch.

The larger it becomes:

* the slower the stream becomes
* the more memory is required

---

# Key Learning from This Lecture

We learned how to inspect streaming queries using Spark UI.

We learned:

* Query monitoring
* Micro-batch statistics
* Processing rates
* Batch duration
* Stateful metrics
* State Store statistics
* Memory usage

Most importantly, we learned:

# State Store growth directly impacts streaming performance

---

# Important Production Insight

Whenever you see:

```text
Increasing batch duration
```

always investigate:

# State Store Size

because large state is one of the most common causes of streaming slowdowns.

---

# Final Summary

---

# Spark Structured Streaming Creates Background Queries

Each query runs as a continuous background thread.

---

# Structured Streaming Tab in Spark UI

Shows:

* running queries
* completed queries
* micro-batch stats
* performance metrics

---

# Important Metrics

| Metric             | Meaning               |
| ------------------ | --------------------- |
| Input Rate         | Incoming records/sec  |
| Processing Rate    | Processed records/sec |
| Batch Duration     | Time per micro-batch  |
| State Rows         | Total state entries   |
| Updated State Rows | Changed state entries |
| State Memory       | Memory used by state  |

---

# Stateful Operations Use State Store

Examples:

* aggregations
* joins
* windows
* deduplication

---

# Large State Store = Performance Risk

As state grows:

* memory usage increases
* processing slows down
* latency increases

---

# Most Important Takeaway

In streaming systems:

# Managing state efficiently is one of the biggest engineering challenges.

And Spark UI is your primary tool for understanding and debugging that state.

---

In the next lecture, we will learn:

* how to control State Store growth
* how to reduce memory usage
* windowing concepts
* watermarking
* handling late-arriving data

These are some of the most advanced and important topics in Spark Structured Streaming.

See you again.

Keep learning and keep growing.


![image_1778499385132.png](./image_1778499385132.png "image_1778499385132.png")
![image_1778501507789.png](./image_1778501507789.png "image_1778501507789.png")
![image_1778501741317.png](./image_1778501741317.png "image_1778501741317.png")

# Spark Structured Streaming — Stateful vs Stateless Transformations (Professor Style Notes)

Welcome back.

So far in this course, we have learned how to build streaming applications using Spark Structured Streaming.

We learned:

* How Spark Structured Streaming works using **micro-batches**
* How to use `readStream`
* How to use `writeStream`
* Different output modes:

  * Append Mode
  * Complete Mode
  * Update Mode
* How to calculate aggregates
* How to implement incremental aggregation
* How to use `foreachBatch`
* How to implement merge logic

Everything looked simple and straightforward.

In fact, if you already know the Spark DataFrame API, then most Spark Structured Streaming concepts feel very natural because the APIs are almost identical.

But Spark Structured Streaming introduces one very important concept that changes everything:

# Stateless vs Stateful Transformations

This is one of the most important architectural concepts in streaming systems.

If you understand this properly, then:

* you will design better streaming pipelines,
* avoid memory issues,
* and know when Spark can or cannot scale.

So in this lecture, we will deeply understand:

1. What is a state?
2. What is a state store?
3. Stateless transformations
4. Stateful transformations
5. Side effects of stateful operations
6. When to use stateful aggregation
7. When to avoid it
8. How to manage state growth

---

# Understanding the Meaning of “State”

Before understanding stateless and stateful transformations, we must first understand:

> What exactly is a “state”?

Let’s take a simple example.

Suppose we are building a customer rewards system.

We receive invoices continuously from a streaming source.

Our requirement is:

* Calculate total purchases for each customer
* Calculate reward points for each customer

---

# First Customer Purchase

Assume a new customer visits the store.

We receive this invoice:

| Invoice No | Customer ID | Amount |
| ---------- | ----------- | ------ |
| 1          | 101         | 500    |

This invoice enters our streaming application.

Spark reads this invoice using `readStream`.

Then we apply aggregation logic.

We calculate:

* Total Purchase = 500
* Reward Points = 10 (assume 2%)

The output becomes:

| Customer ID | Total Purchase | Rewards |
| ----------- | -------------- | ------- |
| 101         | 500            | 10      |

Everything looks simple so far.

But something important also happens behind the scenes.

---

# State Store Behind the Scene

Spark Structured Streaming secretly stores this aggregation result inside something called:

# State Store

The state store now contains:

| Customer ID | Total Purchase | Rewards |
| ----------- | -------------- | ------- |
| 101         | 500            | 10      |

This information is stored:

* in executor memory,
* and also persisted in the checkpoint directory.

Why?

Because Spark must remember previous aggregates.

Without memory of previous results, incremental aggregation is impossible.

---

# Second Purchase by Same Customer

Now suppose the same customer returns later.

New invoice:

| Invoice No | Customer ID | Amount |
| ---------- | ----------- | ------ |
| 2          | 101         | 300    |

Now Spark reads ONLY this new invoice.

Remember:
Spark Structured Streaming always performs incremental reads.

So current micro-batch sees only:

| Customer ID | Amount |
| ----------- | ------ |
| 101         | 300    |

Now if Spark simply calculates sum on this micro-batch, answer should be:

300

But actual output becomes:

| Customer ID | Total Purchase | Rewards |
| ----------- | -------------- | ------- |
| 101         | 800            | 16      |

How?

This is where the state store becomes powerful.

---

# How Spark Calculates Incremental Aggregates

Spark does this internally:

Current Micro-batch Total:

* 300

Previous Total from State Store:

* 500

Final Total:

* 500 + 300 = 800

Similarly:

* previous rewards = 10
* current rewards = 6
* final rewards = 16

Then Spark updates the state store:

| Customer ID | Total Purchase | Rewards |
| ----------- | -------------- | ------- |
| 101         | 800            | 16      |

This process repeats forever.

---

# Third Purchase

Now customer purchases again:

| Invoice No | Customer ID | Amount |
| ---------- | ----------- | ------ |
| 3          | 101         | 800    |

Current batch total:

* 800

Previous total from state:

* 800

Final total:

* 1600

Final rewards:

* 32

Again Spark updates state store automatically.

And all this happens without us writing any special code.

That is the power of Spark Structured Streaming.

---

# Important Realization

In our code, we never wrote:

* read previous totals
* add previous totals
* maintain historical aggregates

We simply wrote:

```python
groupBy("customer_id").agg(sum("amount"))
```

That’s all.

Spark handled:

* previous totals,
* state management,
* checkpointing,
* recovery,
* aggregation continuity.

Automatically.

---

# Stateless Transformations

Now let us properly define stateless transformations.

# Definition

A stateless transformation depends ONLY on current input data.

It does NOT require:

* previous micro-batch data,
* historical information,
* or stored state.

---

# Examples of Stateless Transformations

These are stateless:

* `select`
* `filter`
* `map`
* `flatMap`
* `explode`
* projections
* column transformations

Example:

```python
df.select("customer_id", "amount")
```

This transformation only works on current records.

It does not care:

* what happened yesterday,
* what happened in previous micro-batches,
* or historical aggregates.

Therefore:

* no state store is required,
* no memory persistence is required.

---

# Stateful Transformations

Now let’s define stateful transformations.

# Definition

A stateful transformation requires historical information across micro-batches.

It must remember past results.

Therefore Spark creates and manages a state store behind the scenes.

---

# Examples of Stateful Transformations

These are stateful:

* `groupBy`
* aggregations
* windowing
* streaming joins
* session windows

Why?

Because these operations need past information.

Example:

```python
groupBy("customer_id").sum("amount")
```

To calculate the latest total,
Spark must know:

* previous totals,
* current totals,
* and merge them together.

Therefore:

* Spark creates a state store,
* stores aggregates,
* updates them continuously.

---

# Difference Between Stateless and Stateful

| Feature                | Stateless | Stateful     |
| ---------------------- | --------- | ------------ |
| Needs historical data  | No        | Yes          |
| Uses state store       | No        | Yes          |
| Memory usage           | Low       | Higher       |
| Complexity             | Simple    | More complex |
| Supports complete mode | No        | Yes          |
| Risk of OOM            | Low       | High         |

---

# Important Side Effect of Stateless Transformations

Stateless transformations DO NOT support:

# Complete Output Mode

Why?

Because complete mode requires complete historical results.

But stateless operations do not remember history.

Therefore Spark throws an exception if you use:

```python
outputMode("complete")
```

with stateless transformations.

---

# Important Side Effect of Stateful Transformations

Stateful transformations create state.

State means:

* memory usage,
* checkpoint files,
* executor memory consumption.

And this introduces risk.

---

# The Biggest Problem — State Growth

Suppose you have:

* 1 million customers
* continuous aggregation forever

Then your state store may contain:

1 million rows.

This state:

* remains in memory,
* grows continuously,
* consumes executor RAM.

Eventually:

* performance degrades,
* garbage collection increases,
* executor memory pressure rises,
* OutOfMemory exceptions may occur.

This is one of the biggest real-world streaming problems.

---

# Managing State is a Critical Design Decision

Whenever you design streaming applications, always ask:

> How large can my state become?

Because state size directly affects:

* memory,
* performance,
* scalability,
* stability.

---

# Two Approaches for Aggregation

There are two major ways to implement aggregation.

---

# Approach 1 — Managed Stateful Aggregation

In this approach:

* Spark manages state store
* Spark maintains aggregates
* Spark handles checkpointing
* Spark handles recovery

You simply write aggregation logic.

Example:

```python
groupBy().agg()
```

This is simple and elegant.

But state can grow very large.

---

# Approach 2 — Custom Stateless Aggregation

In this approach:

* You avoid Spark-managed state
* You implement your own incremental logic
* You manually merge results
* You control storage yourself

This is more scalable for certain use cases.

Especially when:

* state size becomes huge,
* or aggregation is unbounded.

---

# Two Types of Aggregation

Now the most important architectural concept.

Aggregations are broadly of two types:

1. Unbounded Continuous Aggregation
2. Time-Bound Aggregation

---

# Unbounded Continuous Aggregation

Example requirement:

> Calculate lifetime total purchases for every customer.

This aggregation never ends.

Totals keep growing forever.

Problems:

* state store grows forever,
* memory keeps increasing,
* scalability becomes difficult.

If you have millions of customers,
state size becomes enormous.

---

# Time-Bound Aggregation

Now modify the requirement:

> Calculate monthly customer rewards.

This changes everything.

Now aggregation has a boundary:

* one month.

At month end:

* rewards are finalized,
* state becomes invalid,
* Spark can clean old state.

Then next month:

* aggregation restarts from zero.

This dramatically reduces state size.

---

# Benefits of Time-Bound Aggregation

## Benefit 1 — Smaller State

Only active customers for current month remain in state.

Instead of:

* 1 million total customers,

you may maintain:

* only 50,000 active customers.

Huge improvement.

---

## Benefit 2 — State Cleanup

Spark now understands:

* old state is expired,
* aggregation window is completed.

So Spark can safely remove old state.

This is called:

# State Store Cleanup

Very important concept.

---

# When to Use Stateful Aggregation

Use managed stateful aggregation when:

* aggregation is time-bound,
* state can expire,
* state size remains manageable.

Examples:

* hourly aggregation
* daily metrics
* monthly rewards
* session windows
* bounded window calculations

---

# When to Avoid Stateful Aggregation

Avoid it when:

* aggregation is unbounded,
* state grows forever,
* millions of keys accumulate.

Instead:

* implement custom stateless aggregation,
* use merge statements,
* manage persistence yourself.

---

# Key Interview Insight

If interviewer asks:

> Why are stateful streaming operations expensive?

Your answer:

Because Spark must:

* maintain historical state,
* store it in memory,
* persist it in checkpoint storage,
* reload it after failures,
* and continuously update it across micro-batches.

State growth directly impacts:

* executor memory,
* performance,
* and scalability.

---

# Final Summary

## Stateless Transformations

* Work only on current data
* No historical memory
* No state store
* Lightweight and scalable

Examples:

* select
* filter
* map
* explode

---

## Stateful Transformations

* Need historical information
* Use state store
* Maintain aggregates across micro-batches

Examples:

* groupBy
* aggregation
* windowing
* streaming joins

---

## Stateful Aggregation Risks

* High memory usage
* State growth
* Performance degradation
* OutOfMemory exceptions

---

## Best Practice

### Use Stateful Aggregation When:

* aggregation is time-bound,
* state can expire.

### Use Custom Stateless Aggregation When:

* aggregation is unbounded,
* state grows forever.

---

# Important Concept to Remember

Spark Structured Streaming processes:

* only new data,

BUT still calculates:

* complete correct aggregates,

because it secretly maintains:

* state store.

That is the magic behind incremental aggregation.

---

That’s all for this lecture.

In the next lecture, we will learn how to implement custom stateless aggregation logic and avoid Spark-managed state for large-scale continuous aggregations.

See you again.

Keep learning and keep growing.


![image_1778502338301.png](./image_1778502338301.png "image_1778502338301.png")
![image_1778502350279.png](./image_1778502350279.png "image_1778502350279.png")
![image_1778502364830.png](./image_1778502364830.png "image_1778502364830.png")
![image_1778502379935.png](./image_1778502379935.png "image_1778502379935.png")
![image_1778502394191.png](./image_1778502394191.png "image_1778502394191.png")

#start changes


![image_1778502503157.png](./image_1778502503157.png "image_1778502503157.png")
![image_1778502553870.png](./image_1778502553870.png "image_1778502553870.png")
![image_1778502559588.png](./image_1778502559588.png "image_1778502559588.png")
![image_1778502674504.png](./image_1778502674504.png "image_1778502674504.png")

# Spark Structured Streaming — Stateless vs Stateful Aggregation

## Implementing Unbounded Incremental Aggregates Without State Store

*(Professor-Style Lecture Notes for Long-Term Memory Retention)*

---

# Introduction — Why This Lecture Matters

Welcome back.

So far in this course, we have learned how to build streaming applications using Spark Structured Streaming.

We learned:

* How to read streaming data using `readStream`
* How to process data incrementally
* How micro-batches work
* How to write results using `writeStream`
* How to calculate aggregates
* How to implement incremental aggregation using:

  * `update` output mode
  * `foreachBatch`
  * `MERGE` statements

Everything looked clean and simple.

But now we are entering one of the most important architectural topics in Spark Structured Streaming:

# **How to calculate aggregates WITHOUT using State Store**

This lecture is extremely important because:

* Stateful aggregation is powerful
* But it can become expensive
* Memory usage can grow infinitely
* Long-running streams may eventually fail with Out Of Memory errors

So in this lecture, we will learn:

* How to implement aggregation without State Store
* How to build **stateless incremental aggregation**
* Why this approach is useful for **unbounded continuous aggregation**
* How to redesign aggregation logic using `MERGE`

This is an advanced real-world design technique.

---

# Part 1 — Problem with Stateful Aggregation

Let us first understand the problem.

In previous lectures, we implemented:

* Customer total purchases
* Customer reward points

And the aggregation was:

```python
groupBy(customer_id).agg(sum(total_amount))
```

This aggregation was happening:

```python
readStream --> transformations --> writeStream
```

Because aggregation existed between:

* `readStream`
* `writeStream`

Spark automatically treated it as:

# Stateful Streaming Aggregation

Which means Spark created:

# State Store

---

# What Does State Store Do?

State Store keeps:

* Previous aggregates
* Intermediate results
* Historical state across micro-batches

Example:

| Customer | Previous Total |
| -------- | -------------- |
| 101      | 500            |

New micro-batch arrives:

| Customer | New Purchase |
| -------- | ------------ |
| 101      | 300          |

Spark combines:

```text
500 + 300 = 800
```

This is possible because State Store remembers history.

---

# The Real Problem

This approach works beautifully.

But what happens if:

* Stream runs forever
* Customers keep increasing
* State keeps growing forever

Then:

* State Store size increases
* Executor memory increases
* Checkpoint size increases
* Disk IO increases
* Recovery becomes slower
* Eventually performance degrades

This is called:

# Unbounded Continuous Aggregation

Meaning:

```text
Aggregation never expires
```

Example:

```text
Calculate total purchases forever
```

No reset.

No boundary.

No expiry.

Very dangerous for State Store growth.

---

# Part 2 — The Goal of This Lecture

Our goal is:

# Calculate aggregates WITHOUT using State Store

Meaning:

* No streaming aggregation between read/write
* No Spark-managed state
* No executor memory overhead
* No checkpoint state growth

But still:

* Correct incremental totals
* Correct cumulative aggregation
* Same final output

This is called:

# Stateless Incremental Aggregation

---

# Part 3 — High-Level Idea

The trick is extremely clever.

Instead of doing aggregation:

```python
readStream --> aggregate --> writeStream
```

We do:

```python
readStream --> writeStream --> aggregate inside foreachBatch
```

This changes EVERYTHING.

---

# Why Does This Remove State Store?

Spark only creates State Store when aggregation exists:

```python
BETWEEN readStream and writeStream
```

If aggregation is removed from streaming transformations:

```python
No stateful operation
```

Then Spark treats it as:

# Stateless Stream

No State Store is created.

---

# Part 4 — Starting with Existing Example

We already had this application:

# Bronze Layer

Reads invoices from landing zone and stores raw data.

# Gold Layer

Reads invoices and calculates:

* Customer total sales
* Customer reward points

Previously this was stateful.

Now we want to convert it into:

# Stateless Incremental Aggregation

---

# Part 5 — Existing Bronze Layer

The Bronze Layer remains unchanged.

---

## Bronze Layer Responsibility

```text
Landing Zone --> Bronze Table
```

No transformations.

Just ingestion.

---

## Bronze Layer Logic

### Read invoices

```python
spark.readStream
```

### Write invoices

```python
writeStream.toTable()
```

Simple ingestion pipeline.

No changes required.

---

# Part 6 — Existing Gold Layer (Old Stateful Approach)

Previously we had:

```python
readStream
    --> groupBy()
    --> agg()
    --> writeStream
```

Example:

```python
groupBy("customer_card_no")
.agg(sum("total_amount"))
```

This created:

# Stateful Aggregation

And Spark created:

# State Store

Automatically.

---

# Part 7 — The New Stateless Design

Now we redesign the architecture.

Instead of:

```python
read --> aggregate --> write
```

We will do:

```python
read --> write --> aggregate in foreachBatch
```

This is the core idea.

---

# Part 8 — Remove Aggregation from Streaming Pipeline

Previously:

```python
aggregates = self.get_aggregates(invoices_df)
```

We REMOVE this line.

Why?

Because this aggregation exists inside streaming flow.

That triggers State Store creation.

---

# New Process Flow

Now process becomes:

```python
invoices_df = self.read_bronze()

self.save_results(invoices_df)
```

Notice:

* No aggregation
* No groupBy
* No stateful transformation

So Spark sees:

# Stateless Stream

---

# Part 9 — Using foreachBatch

Now we move aggregation into:

```python
foreachBatch()
```

Why?

Because inside `foreachBatch`:

* We receive one micro-batch as a normal DataFrame
* We can execute custom logic
* We can use SQL
* We can use MERGE
* We can manually calculate aggregates

Most importantly:

# Spark does NOT create State Store here

---

# Part 10 — Changing Callback Method

Previously:

```python
foreachBatch(self.upsert)
```

Now:

```python
foreachBatch(self.aggregate_upsert)
```

Why?

Because callback must now:

1. Calculate aggregates
2. Merge results

Both responsibilities moved here.

---

# Part 11 — Understanding What foreachBatch Receives

Important concept.

The callback receives:

# Only current micro-batch data

Meaning:

```python
invoices_df
```

contains:

```text
ONLY NEW RECORDS
```

Not historical data.

Not previous invoices.

Only incremental invoices.

---

# Part 12 — Aggregate Inside Callback

Now inside callback:

```python
rewards_df = self.get_aggregates(invoices_df)
```

We calculate:

* Current micro-batch totals
* Current micro-batch rewards

Example:

| Customer | Current Batch Total |
| -------- | ------------------- |
| 101      | 300                 |

This is NOT final cumulative total yet.

---

# Part 13 — The Big Challenge

Now comes the important question.

If current batch total is:

```text
300
```

But historical total was:

```text
500
```

How do we produce:

```text
800
```

Without State Store?

---

# Part 14 — The Secret Trick

We use:

# Target Table as State Store

This is the genius idea.

Instead of Spark State Store:

```text
Checkpoint State Store
```

We use:

```text
Customer Rewards Table
```

as our state repository.

---

# Part 15 — MERGE Logic

Now the merge becomes smarter.

Previously:

```sql
UPDATE target.total_amount = source.total_amount
```

Now:

```sql
UPDATE target.total_amount =
       source.total_amount + target.total_amount
```

This changes everything.

---

# What Is Happening Here?

Suppose:

## Existing Table

| Customer | Total |
| -------- | ----- |
| 101      | 500   |

## Current Batch Aggregate

| Customer | Total |
| -------- | ----- |
| 101      | 300   |

Merge logic becomes:

```text
500 + 300 = 800
```

Then update table.

---

# This Is Equivalent to State Store

Spark previously did:

```text
State Store + Current Batch
```

Now WE do:

```text
Target Table + Current Batch
```

Same logic.

Same output.

WITHOUT State Store.

---

# Part 16 — Final MERGE Logic

---

## When Matched

Existing customer:

```sql
target.total_amount =
    target.total_amount + source.total_amount
```

Similarly:

```sql
target.total_points =
    target.total_points + source.total_points
```

---

## When Not Matched

New customer:

```sql
INSERT *
```

Simple.

---

# Part 17 — Why This Approach Is Powerful

This approach gives multiple benefits.

---

## Benefit 1 — No State Store

Spark does NOT maintain:

* executor memory state
* checkpoint state
* aggregation metadata

Huge savings.

---

## Benefit 2 — Better Memory Usage

No growing in-memory state.

Memory remains stable.

---

## Benefit 3 — Better Scalability

Works well for:

* unbounded streams
* infinite aggregations
* very large customer bases

---

## Benefit 4 — Simpler State Management

Instead of Spark state:

```text
Target table becomes truth
```

Easy to manage.

Easy to debug.

Easy to recover.

---

# Part 18 — Important Architectural Insight

This is one of the most important ideas in streaming design.

---

# Stateful Aggregation

```text
Spark manages state
```

Good for:

* bounded aggregation
* windowing
* time-based logic

---

# Stateless Aggregation

```text
Application manages state
```

Good for:

* unbounded aggregation
* continuous totals
* forever-running metrics

---

# Part 19 — What Still Happens Incrementally?

Even without State Store:

Spark still gives:

# Incremental Reading

Because:

```python
readStream
```

still reads only new data.

So micro-batch processing remains incremental.

Only aggregation responsibility moved outside Spark state engine.

---

# Part 20 — Test Suite Changes

Most test logic remained unchanged.

---

## Important Change

Previously:

```python
enable RocksDB state store
```

Now removed.

Why?

Because:

# We no longer use State Store

So RocksDB optimization becomes unnecessary.

---

# Part 21 — One Important Fix

Since we use:

```sql
MERGE INTO customer_rewards
```

The target table must already exist.

So in cleanup phase:

```python
CREATE TABLE customer_rewards
```

was added.

Otherwise MERGE fails.

---

# Part 22 — Spark UI Observation

When running the query:

You still see:

* Bronze query
* Gold query

inside Spark UI.

But now:

# No Stateful Aggregation Metrics

Because:

* No State Store
* No aggregation state
* No executor state memory

Huge architectural difference.

---

# Part 23 — Final Mental Model

---

# Old Stateful Flow

```text
readStream
    ↓
groupBy + agg
    ↓
Spark State Store
    ↓
writeStream
```

Problems:

* Growing memory
* Growing checkpoint
* Executor overhead

---

# New Stateless Flow

```text
readStream
    ↓
writeStream
    ↓
foreachBatch
    ↓
aggregate current batch
    ↓
MERGE with target table
```

Benefits:

* No State Store
* Better scalability
* Better for unbounded aggregation

---

# Part 24 — When Should You Use This?

Use this approach when:

* Aggregation never expires
* Stream runs forever
* State grows infinitely
* Customer count becomes massive
* Stateful aggregation becomes dangerous

Examples:

* Lifetime customer spend
* Lifetime reward points
* Total clicks forever
* Running account balances
* Continuous counters

---

# Part 25 — Key Interview Insight

Very important.

---

# Spark Stateful Aggregation

Best for:

* time windows
* event-time processing
* watermarking
* bounded state

---

# Custom Stateless Aggregation

Best for:

* unbounded aggregation
* infinite running totals
* memory-sensitive systems

This distinction is extremely important in real-world streaming architecture interviews.

---

# Final Summary

In this lecture we learned:

---

## What Problem We Solved

We wanted:

```text
Aggregation WITHOUT State Store
```

---

## Key Idea

Move aggregation:

```text
AFTER writeStream
```

inside:

```python
foreachBatch()
```

---

## Core Trick

Use:

```text
Target Table as State Store
```

instead of Spark-managed state.

---

## Key Benefits

* No executor memory overhead
* No growing State Store
* Better scalability
* Better for unbounded aggregation

---

# Final One-Line Memory Trick

# “Stateful aggregation stores history in Spark memory. Stateless aggregation stores history in the target table.”

---

# End of Lecture

That’s all for this lecture.

In the next lecture, we will continue exploring advanced streaming architectures and optimization techniques.

Keep learning and keep growing.


![image_1778507758852.png](./image_1778507758852.png "image_1778507758852.png")
![image_1778507743390.png](./image_1778507743390.png "image_1778507743390.png")
![image_1778508788410.png](./image_1778508788410.png "image_1778508788410.png")

![image_1778508854584.png](./image_1778508854584.png "image_1778508854584.png")
![image_1778508912387.png](./image_1778508912387.png "image_1778508912387.png")
![image_1778508924664.png](./image_1778508924664.png "image_1778508924664.png")
![image_1778508965347.png](./image_1778508965347.png "image_1778508965347.png")
![image_1778509040661.png](./image_1778509040661.png "image_1778509040661.png")
![image_1778509066916.png](./image_1778509066916.png "image_1778509066916.png")
![image_1778509075391.png](./image_1778509075391.png "image_1778509075391.png")
![image_1778509088504.png](./image_1778509088504.png "image_1778509088504.png")
![image_1778509113822.png](./image_1778509113822.png "image_1778509113822.png")

# Time-Bound Aggregation in Spark Structured Streaming

## Understanding Window Aggregates Like a Real-Time Analytics Engineer

Welcome back.

So far, we learned how to calculate incremental aggregates using Spark Structured Streaming. We learned about stateful aggregations, stateless aggregations, and how Spark maintains state internally using the state store.

In this lecture, we are going to move to one of the most important concepts in real-time analytics systems:

* **Time Bound Aggregation**
* **Window Aggregation**
* **Time Window Based Aggregation**

All these names refer to the same idea.

This concept is extremely important because almost every real-time analytics system in the industry depends on windowing logic.

Examples include:

* Stock market dashboards
* Website traffic analytics
* Fraud detection systems
* IoT sensor monitoring
* Banking transaction monitoring
* Real-time sales dashboards
* Streaming telemetry systems

So this lecture is very important from both interview and real-world project perspectives.

---

# 1. What is Time Bound Aggregation?

Let us start with the simplest possible definition.

A **time-bound aggregation** means:

> We calculate aggregates only for a fixed period of time.

That fixed duration is called a **window**.

The window can be:

* milliseconds
* seconds
* minutes
* hours
* days
* weeks

Examples:

* Total sales every 5 minutes
* Website clicks every 1 hour
* Total stock trades every 15 minutes
* Total fraud attempts every 30 seconds

So instead of calculating one infinite continuous aggregate forever, we divide the stream into smaller time buckets.

---

# 2. Why Do We Need Window Aggregation?

Think carefully.

Suppose a stock trading company wants to build a dashboard.

Their users are continuously buying and selling stocks.

Now management wants a dashboard that refreshes every 15 minutes and shows:

* Total Buy Amount
* Total Sell Amount
* Net Position

Can we calculate one infinite aggregate forever?

No.

Because dashboard users are interested in trends over time.

They want answers like:

* What happened between 10:00 and 10:15?
* What happened between 10:15 and 10:30?
* What happened between 10:30 and 10:45?

This is exactly where window aggregation becomes useful.

---

# 3. Business Scenario — Real-Time Stock Trading Analytics

Let us understand the complete scenario.

Imagine you are working for a stock brokerage company.

Customers use:

* mobile apps
* trading terminals
* websites

to buy and sell stocks.

Every trade transaction is immediately pushed into a Kafka topic.

---

## Architecture Flow

The architecture looks like this:

```text
Trading App
     ↓
Kafka Topic
     ↓
Kafka Bronze Table
     ↓
Structured Streaming Job
     ↓
Trade Summary Table
     ↓
Dashboard / Visualization
```

---

# 4. Trade Transaction Data

Every trade event arrives as a JSON message.

Example:

```json
{
  "CreatedTime": "2022-01-01 10:05:00",
  "Type": "BUY",
  "Amount": 500,
  "BrokerCode": "ABX"
}
```

Fields:

| Field       | Meaning             |
| ----------- | ------------------- |
| CreatedTime | When trade happened |
| Type        | BUY or SELL         |
| Amount      | Trade amount        |
| BrokerCode  | Broker identifier   |

---

# 5. Kafka Bronze Table Structure

When data is ingested from Kafka and saved into the Bronze layer, the table looks like this:

| Key        | Value               |
| ---------- | ------------------- |
| 2022-01-01 | Entire JSON message |

The `value` column contains the complete JSON event as a string.

---

# 6. Final Requirement

We want to create a **Trade Summary Table**.

The table should contain:

| Start | End | TotalBuy | TotalSell |
| ----- | --- | -------- | --------- |

Where:

* Start = Window start time
* End = Window end time
* TotalBuy = Sum of BUY transactions in that window
* TotalSell = Sum of SELL transactions in that window

---

# 7. Understanding the 15-Minute Window

Suppose the window size is 15 minutes.

Then within one hour, Spark automatically creates:

| Window Start | Window End |
| ------------ | ---------- |
| 10:00        | 10:15      |
| 10:15        | 10:30      |
| 10:30        | 10:45      |
| 10:45        | 11:00      |

Now every event is placed into one of these windows based on its timestamp.

---

# 8. Example of Window Assignment

Suppose a trade happened at:

```text
10:05
```

Spark checks:

```text
Does 10:05 belong to 10:00–10:15?
```

Yes.

So Spark places that record into the first window.

---

Another event occurs at:

```text
10:20
```

Spark places it into:

```text
10:15–10:30
```

This is how window grouping works internally.

---

# 9. Important Requirement for Window Aggregation

To implement time-window aggregation, your dataset MUST contain a timestamp column.

Without timestamp information:

* Spark cannot determine event timing
* Spark cannot place events into windows

In this example:

```text
CreatedTime
```

is our event-time column.

---

# 10. Designing the Streaming Application

We already know how to:

* read stream
* transform data
* write stream

Now we focus specifically on window aggregation.

---

# 11. Creating the Streaming Class

We create a class:

```python
class TradeSummary:
```

This class contains:

* schema definition
* read logic
* transformation logic
* aggregation logic
* write logic

---

# 12. Creating JSON Schema

The Kafka value column contains JSON.

So we first define schema:

```python
StructType([
    StructField("CreatedTime", StringType()),
    StructField("Type", StringType()),
    StructField("Amount", DoubleType()),
    StructField("BrokerCode", StringType())
])
```

---

# 13. Reading Stream from Bronze Table

We use:

```python
spark.readStream.table("kafkabz")
```

This continuously reads incremental data from the Kafka Bronze table.

---

# 14. Transforming JSON into Structured Columns

The Kafka value column is a JSON string.

We convert it into proper columns using:

```python
from_json()
```

Example:

```python
from_json(col("value"), schema)
```

This converts raw JSON into structured columns.

---

# 15. Expanding Struct Fields

After parsing JSON, we extract fields using:

```python
select("value.*")
```

Now we get:

| CreatedTime | Type | Amount | BrokerCode |

---

# 16. Converting String to Timestamp

Initially:

```text
CreatedTime
```

is a string.

But Spark window function requires proper timestamp datatype.

So we convert it:

```python
to_timestamp()
```

---

# 17. Why We Create BUY and SELL Columns

Initially we have:

| Type | Amount |
| ---- | ------ |
| BUY  | 500    |
| SELL | 400    |

Aggregation becomes difficult.

Instead, we transform data into:

| Buy | Sell |
| --- | ---- |
| 500 | 0    |
| 0   | 400  |

Now aggregation becomes extremely easy.

We can simply calculate:

```python
sum(Buy)
sum(Sell)
```

---

# 18. Using CASE Expression

We implement transformation like this:

```sql
CASE WHEN Type='BUY' THEN Amount ELSE 0 END
```

Similarly for SELL.

This creates two clean numeric columns.

---

# 19. Final Trade DataFrame

After transformation, our DataFrame becomes:

| CreatedTime | Buy | Sell |
| ----------- | --- | ---- |
| 10:05       | 500 | 0    |
| 10:12       | 300 | 0    |
| 10:20       | 600 | 0    |
| 10:25       | 0   | 400  |

Now aggregation is extremely simple.

---

# 20. Core Concept — Window Function

Now comes the most important concept of this lecture.

Spark provides a special function:

```python
window()
```

This function creates time windows.

Syntax:

```python
window(timestamp_column, window_duration)
```

---

# 21. Our Window Aggregation Logic

We write:

```python
groupBy(
    window(col("CreatedTime"), "15 minutes")
)
```

This means:

> Group all records into 15-minute time windows.

---

# 22. Internally What Spark Does

Spark automatically creates windows like:

| Start | End   |
| ----- | ----- |
| 10:00 | 10:15 |
| 10:15 | 10:30 |
| 10:30 | 10:45 |

Then Spark checks each record’s timestamp and places it into the correct bucket.

---

# 23. Window Aggregation Logic

After grouping, aggregation becomes easy.

We calculate:

```python
sum("Buy")
sum("Sell")
```

So final aggregation becomes:

```python
groupBy(window(...))
.agg(
    sum("Buy").alias("TotalBuy"),
    sum("Sell").alias("TotalSell")
)
```

---

# 24. Structure of Window Column

The `window()` function creates a special struct column.

It contains:

| Field | Meaning                |
| ----- | ---------------------- |
| start | Window start timestamp |
| end   | Window end timestamp   |

So we later extract:

```python
window.start
window.end
```

---

# 25. Final Aggregated Result

The result looks like:

| Start | End   | TotalBuy | TotalSell |
| ----- | ----- | -------- | --------- |
| 10:00 | 10:15 | 800      | 0         |
| 10:15 | 10:30 | 600      | 400       |
| 10:30 | 10:45 | 900      | 0         |
| 10:45 | 11:00 | 0        | 500       |

---

# 26. Understanding Late Arriving Events

Now comes another extremely important concept.

Suppose:

Event Time:

```text
10:25
```

But it reaches the system late at:

```text
10:40
```

Can Spark handle this?

Yes.

Spark Structured Streaming supports late-arriving records.

---

# 27. What Happens Internally?

Spark checks event time:

```text
10:25
```

Even if the event arrived late physically, Spark places it into:

```text
10:15–10:30
```

window.

Then Spark recalculates aggregates for that window.

This is extremely powerful.

---

# 28. Why Event Time Matters

Spark always uses:

```text
Event Time
```

not arrival time for window calculations.

This is very important.

---

# 29. Writing Results to Output Table

Finally we use:

```python
writeStream
```

and save results into:

```text
TradeSummary
```

table.

---

# 30. Why Complete Output Mode is Used

For window aggregation, we often use:

```python
outputMode("complete")
```

Because every microbatch may update previous window results.

Especially when late events arrive.

---

# 31. Testing the Application

We create a test suite.

The testing strategy is:

```text
Insert Events
↓
Wait for Microbatch
↓
Validate Aggregates
```

This is the standard streaming testing methodology.

---

# 32. Carefully Designed Test Data

We intentionally designed events like:

| Time  | Type | Amount |
| ----- | ---- | ------ |
| 10:05 | BUY  | 500    |
| 10:12 | BUY  | 300    |
| 10:20 | BUY  | 600    |
| 10:25 | SELL | 400    |
| 10:40 | BUY  | 900    |
| 10:48 | SELL | 500    |

This allows us to test:

* multiple windows
* late events
* buy/sell aggregation

---

# 33. Expected Output

We manually calculate expected results.

This is very important in streaming tests.

Always calculate expected aggregates manually.

---

# 34. Visualization and Dashboard

Once data exists in the TradeSummary table:

Dashboard teams can build:

* line charts
* bar charts
* trend graphs
* real-time dashboards

Example:

* Blue Line → BUY
* Red Line → SELL
* Green Line → NET

---

# 35. Big Picture Understanding

The complete flow is:

```text
Kafka Events
↓
Structured Streaming
↓
Window Aggregation
↓
Trade Summary Table
↓
Dashboard
```

This is a very common enterprise streaming architecture.

---

# 36. Most Important Learning From This Lecture

## Window aggregation is simply:

> Grouping records by time buckets.

That’s all.

The magic is done by:

```python
window(timestamp, duration)
```

---

# 37. Key Interview Concepts

You must remember these interview points:

| Concept            | Important Point             |
| ------------------ | --------------------------- |
| Window Aggregation | Time-based grouping         |
| Event Time         | Used for window calculation |
| Window Function    | Creates time buckets        |
| Late Events        | Supported                   |
| Timestamp Column   | Mandatory                   |
| Complete Mode      | Commonly used               |
| Window Struct      | Contains start and end      |

---

# 38. Final Mental Model

Think of window aggregation like this:

Instead of calculating one infinite aggregate:

```text
Total Sales Forever
```

we divide time into buckets:

```text
10:00–10:15
10:15–10:30
10:30–10:45
```

and calculate aggregates separately for each bucket.

That is the essence of time-bound aggregation.

---

# Final Summary

In this lecture we learned:

* What is time-bound aggregation
* What is window aggregation
* Why timestamp is required
* How Spark window() function works
* How to build 15-minute windows
* How records are grouped into windows
* How to calculate buy/sell totals
* How late-arriving records are handled
* How dashboards consume streaming aggregates
* How to test window aggregations properly

This concept is one of the foundational pillars of real-time streaming systems.

In the next lecture, we will continue discussing:

* late-arriving events
* watermarking
* state cleanup
* advanced windowing strategies

These concepts are extremely important for production-grade streaming systems.

See you again in the next lecture.

Keep learning and keep growing.


![image_1778514458165.png](./image_1778514458165.png "image_1778514458165.png")
    # Stateful vs Stateless Transformations in Spark Structured Streaming

## Understanding State Store, Unbounded Aggregations, Time Windows, and Watermarks

Welcome back.

So far in our Spark Structured Streaming journey, we have learned how to:

* Read streaming data using `readStream`
* Apply transformations
* Perform aggregations
* Write results using `writeStream`
* Build incremental streaming pipelines

Everything looked simple and straightforward because Spark Structured Streaming uses the same DataFrame API concepts we already know from batch processing.

However, Spark Structured Streaming introduces one extremely important concept:

# Stateful vs Stateless Transformations

This concept is the backbone of streaming applications.

If you truly understand:

* Stateful processing
* Stateless processing
* State Store
* Window Aggregation
* Watermarks

then you understand the heart of Structured Streaming.

So in these lecture notes, we will deeply understand:

1. What is state?
2. Stateless transformations
3. Stateful transformations
4. State Store
5. Unbounded aggregations
6. Stateless custom aggregation
7. Time window aggregations
8. Watermarks
9. State cleanup

---

# 1. Understanding the Idea of “State”

Before understanding stateful processing, we must first understand:

> What exactly is a state?

Spark Structured Streaming works using **micro-batches**.

Each micro-batch:

1. Reads new streaming data
2. Applies transformations
3. Produces output

Simple.

But some operations need information from previous micro-batches.

That remembered information is called:

# State

---

# Example: Customer Purchase Aggregation

Assume we are building a streaming application for a retail store.

We want to calculate:

* Customer total purchases
* Customer reward points

---

## First Purchase

A customer enters the store.

Invoice:

| Invoice | Customer | Amount |
| ------- | -------- | ------ |
| 1       | 101      | 500    |

Spark reads this invoice.

Then aggregation logic calculates:

| Customer | Total Purchase | Rewards |
| -------- | -------------- | ------- |
| 101      | 500            | 10      |

Now here is the important part.

Spark not only produces output.

It also secretly stores this result internally.

That hidden storage is called:

# State Store

So internally Spark remembers:

| Customer | Purchase | Rewards |
| -------- | -------- | ------- |
| 101      | 500      | 10      |

---

# Why Does Spark Need This State?

Because the customer may come again later.

---

## Second Purchase

Next day:

| Invoice | Customer | Amount |
| ------- | -------- | ------ |
| 2       | 101      | 300    |

Current micro-batch only contains:

| Customer | Amount |
| -------- | ------ |
| 101      | 300    |

But final output becomes:

| Customer | Total Purchase | Rewards |
| -------- | -------------- | ------- |
| 101      | 800            | 16      |

How?

Because Spark:

1. Reads current batch aggregate = 300
2. Reads old value from State Store = 500
3. Combines both

So:

[
500 + 300 = 800
]

This is stateful aggregation.

Spark remembers past results across micro-batches.

---

# Third Purchase

Customer returns again.

| Invoice | Customer | Amount |
| ------- | -------- | ------ |
| 3       | 101      | 800    |

Now:

Previous total = 800
Current purchase = 800

Final:

[
800 + 800 = 1600
]

Rewards update similarly.

Spark continuously maintains cumulative state.

---

# 2. What is State Store?

State Store is Spark’s internal memory mechanism for stateful operations.

It stores:

* Previous aggregates
* Previous joins
* Window states
* Streaming state information

---

# Where is State Store Stored?

State exists in:

1. Executor memory
2. Checkpoint directory

Checkpoint storage ensures fault tolerance.

If the streaming job crashes:

* Spark reloads state from checkpoints
* Processing resumes safely

---

# 3. Stateless Transformations

Now let us understand stateless transformations.

A stateless transformation only works on:

# Current micro-batch data

It does NOT need previous data.

Examples:

* `select`
* `filter`
* `map`
* `flatMap`
* `explode`

These transformations:

* Process current rows
* Produce output
* Forget everything

No memory required.

No state store required.

---

# Example

Suppose a micro-batch has 10 records.

A `filter` transformation checks those 10 rows.

It does not care:

* What happened earlier
* What happened in previous batches

Therefore:

# No state is needed

---

# 4. Stateful Transformations

Stateful transformations need:

# Past information

Examples:

* Aggregations
* Window aggregations
* Streaming joins

These operations must remember earlier results.

Therefore Spark automatically creates and manages State Store.

---

# Important Side Effects

Stateful transformations introduce:

* Memory overhead
* State management complexity
* Risk of out-of-memory issues

Because state continuously grows.

---

# 5. Problem with Unbounded Aggregations

Suppose requirement says:

> Calculate customer lifetime purchases forever.

This is called:

# Unbounded Continuous Aggregation

Because:

* Aggregation never ends
* State never expires
* Spark keeps storing customer data forever

If business has:

* 1 million customers

then state store may eventually contain:

* 1 million entries

Huge memory usage.

---

# 6. Time-Bound Aggregations

Now suppose business changes requirement:

> Calculate monthly rewards only.

Now aggregation becomes:

# Time-Bound Aggregation

Meaning:

* January totals valid only for January
* February starts fresh
* Old state can expire

This is much safer.

Because Spark can now:

# Clean old state

This prevents unlimited memory growth.

---

# Key Benefit of Time Bound Aggregation

Instead of storing:

* All customers forever

Spark stores only:

* Active customers within current window

This dramatically reduces memory usage.

---

# 7. Two Approaches for Aggregations

We now have two approaches.

---

# Approach 1: Managed Stateful Aggregation

Use Spark’s built-in aggregation functions.

Example:

```python
groupBy().agg()
```

Spark:

* Creates state store
* Manages state
* Handles aggregation
* Performs cleanup

Easy and powerful.

Best for:

# Time-bound aggregations

---

# Approach 2: Custom Stateless Aggregation

This is advanced.

We avoid Spark state store completely.

Instead:

* Read current batch
* Aggregate current data manually
* Merge results into target table

No state store involved.

Best for:

# Unbounded aggregations

---

# 8. Custom Stateless Aggregation Strategy

This is a very clever technique.

---

# Traditional Stateful Flow

Normally:

```python
readStream
   -> aggregation
      -> writeStream
```

Aggregation happens between read and write.

Therefore Spark creates state store.

---

# Stateless Custom Approach

Instead:

```python
readStream
   -> stateless transforms
      -> writeStream
          -> foreachBatch()
               -> aggregate manually
               -> MERGE into target
```

Now aggregation happens AFTER writeStream inside `foreachBatch`.

Spark no longer sees streaming aggregation.

Therefore:

# No state store is created

---

# Key Trick

Inside `foreachBatch()`:

1. Aggregate current micro-batch
2. Read existing totals from target table
3. Merge old totals + new totals

Essentially:

# Target table becomes our custom state store

---

# Example MERGE Logic

```sql
MERGE INTO customer_rewards t
USING rewards_df s
ON t.customer_id = s.customer_id

WHEN MATCHED THEN
UPDATE SET
t.total_amount = t.total_amount + s.total_amount

WHEN NOT MATCHED THEN
INSERT *
```

This replicates Stateful Aggregation behavior without using Spark state store.

---

# Advantages

* No executor state memory
* No Spark state management overhead
* Better for endless aggregations

---

# 9. Time Window Aggregation

Now let us move to:

# Time Window Aggregation

Also called:

* Window Aggregation
* Time-Bound Aggregation
* Tumbling Windows

---

# Core Idea

Instead of aggregating forever:

We aggregate within:

# Time windows

Example:

* 15 seconds
* 15 minutes
* 1 hour
* 7 days

---

# Real Trading Scenario

Assume a stock trading platform.

Every trade event contains:

| Created Time | Type | Amount |
| ------------ | ---- | ------ |
| 10:05        | BUY  | 500    |

Goal:

Create 15-minute trading summaries.

---

# Required Output

| Start | End   | Total Buy | Total Sell |
| ----- | ----- | --------- | ---------- |
| 10:00 | 10:15 | 800       | 0          |
| 10:15 | 10:30 | 600       | 400        |

Each row represents one:

# Time Window

---

# Window Function

Spark provides the `window()` function.

Example:

```python
groupBy(
    window(col("created_time"), "15 minutes")
)
```

Spark automatically:

* Creates windows
* Assigns records
* Groups data
* Calculates aggregates

---

# Important Requirement

Time-window aggregation requires:

# Timestamp column

Without timestamp:

* Windowing is impossible

---

# Windowing Logic

Suppose records arrive at:

* 10:05
* 10:12

Both belong to:

[
10:00 \rightarrow 10:15
]

So Spark groups them together.

---

# Another Example

Record at:

* 10:20

belongs to:

[
10:15 \rightarrow 10:30
]

Spark automatically identifies correct window.

---

# 10. Late Arriving Records

Now comes a major streaming problem.

Suppose:

* Record for 10:25 arrives very late
* Spark already processed later windows

Should Spark ignore it?

No.

Spark supports:

# Late Arriving Events

It can:

* Retrieve old window state
* Recalculate aggregates
* Update output

Amazing capability.

---

# Problem Introduced by Late Records

Because late records may arrive anytime:

Spark cannot safely delete old state.

It must preserve old windows indefinitely.

This causes:

# State Store Growth

---

# 11. Watermark

This is where watermark solves the problem.

---

# What is Watermark?

Watermark tells Spark:

# How long should we wait for late records?

Example:

```python
.withWatermark("created_time", "30 minutes")
```

Meaning:

> Wait maximum 30 minutes for late data.

After that:

* Window considered expired
* State eligible for cleanup

---

# Watermark + Window Example

Suppose watermark = 30 minutes.

If a window is older than 30 minutes:

Spark may:

# Remove it from State Store

This enables:

# State Cleanup

---

# What Happens if Very Late Record Arrives?

Suppose:

* Window expired
* State already cleaned
* Record arrives after watermark

Spark simply:

# Ignores the record

No error.

No recalculation.

Record is dropped.

---

# 12. State Store Cleanup

Without watermark:

* Spark keeps all windows forever

With watermark:

* Spark safely removes expired windows

This prevents uncontrolled memory growth.

---

# Watermark Placement Rule

Watermark must be applied:

# BEFORE groupBy()

Correct:

```python
df.withWatermark("created_time", "30 minutes") \
  .groupBy(window(col("created_time"), "15 minutes"))
```

---

# Important Rule

Watermark column and window column should be:

# Same timestamp column

---

# 13. How to Decide Watermark?

This is a business decision.

Two common approaches:

---

# Approach 1: Maximum Expected Delay

Ask business:

> What is maximum acceptable delay?

Example:

* 30 minutes
* 1 hour
* 1 day

Use that as watermark.

---

# Approach 2: Business Relevance

Sometimes old data becomes irrelevant.

Example:

Stock market dashboard valid only for current trading day.

After market closes:

* Old late records no longer matter

So watermark can match business relevance window.

---

# Example

Suppose:

99.99% of records arrive within 30 minutes.

Then:

# 30-minute watermark is sufficient

Remaining extremely late records can be ignored safely.

---

# Final Mental Model

# Stateless Transformations

* Work only on current data
* No memory
* No state store

Examples:

* select
* filter
* explode

---

# Stateful Transformations

* Need previous data
* Use state store
* Maintain memory

Examples:

* aggregations
* windows
* joins

---

# Unbounded Aggregation

* Aggregation never ends
* Dangerous state growth
* Prefer custom stateless approach

---

# Time-Bound Aggregation

* Aggregation has expiry
* Safer
* Spark can clean old state

---

# Watermark

Defines:

# How late is “too late”

After watermark expires:

* State cleaned
* Late records ignored

---

# Final Architecture Understanding

## Spark Structured Streaming Internally Works Like This

```text
Input Stream
    ↓
Micro-Batch
    ↓
Window Assignment
    ↓
State Store Lookup
    ↓
Aggregation
    ↓
Output Sink
    ↓
Checkpoint + State Persistence
```

And with watermark:

```text
Old Window Expired
    ↓
Cleanup Triggered
    ↓
State Removed
```

---

# Key Takeaways

## Stateless Processing

* Lightweight
* Fast
* No memory overhead

---

## Stateful Processing

* Powerful
* Supports incremental analytics
* Requires careful memory management

---

## Time Windows

Allow bounded streaming aggregation.

---

## Watermarks

Allow state cleanup and late-record handling.

---

# Final Conclusion

Spark Structured Streaming becomes truly powerful when you understand:

* State
* State Store
* Window Aggregation
* Watermarks
* Late Event Handling

These concepts are the foundation of:

* Real-time dashboards
* Financial analytics
* Fraud detection
* IoT pipelines
* Monitoring systems
* Streaming ETL architectures

Once you master these ideas, you can design production-grade streaming systems confidently.

Keep learning and keep growing.


![image_1778514639516.png](./image_1778514639516.png "image_1778514639516.png")
![image_1778514656447.png](./image_1778514656447.png "image_1778514656447.png")
![image_1778514695699.png](./image_1778514695699.png "image_1778514695699.png")
![image_1778514734305.png](./image_1778514734305.png "image_1778514734305.png")
![image_1778514752543.png](./image_1778514752543.png "image_1778514752543.png")
![image_1778514766294.png](./image_1778514766294.png "image_1778514766294.png")
![image_1778514783740.png](./image_1778514783740.png "image_1778514783740.png")
![image_1778514822642.png](./image_1778514822642.png "image_1778514822642.png")
![image_1778514891171.png](./image_1778514891171.png "image_1778514891171.png")

![image_1778515003113.png](./image_1778515003113.png "image_1778515003113.png")
# Sliding Window Aggregation in Spark Structured Streaming — Professor Style Notes

Welcome back.

In this lecture, we are going to learn about **Sliding Window Aggregation** in Spark Structured Streaming.

In the earlier lectures, we already learned:

* Event Time Processing
* Tumbling Window Aggregates
* Watermarks
* State Store Cleanup

Now we are moving to a more advanced and extremely useful type of streaming aggregation called:

# Sliding Window Aggregation

Sometimes also known as:

* Hopping Windows
* Moving Window Aggregates
* Rolling Aggregates

This concept is very important in:

* Real-time monitoring systems
* Sensor analytics
* Healthcare systems
* Financial monitoring
* Fraud detection
* IoT analytics
* Machine monitoring systems

So let us understand it step by step.

---

# 1. Quick Revision — Tumbling Windows

Before understanding sliding windows, let us quickly revise tumbling windows.

In tumbling windows:

* Every window has a fixed size
* Windows do NOT overlap
* Each event belongs to exactly ONE window

Example:

If the window size is 15 minutes:

| Window Start | Window End |
| ------------ | ---------- |
| 10:00        | 10:15      |
| 10:15        | 10:30      |
| 10:30        | 10:45      |

Notice carefully:

* Windows are fixed
* Windows are adjacent
* No overlap exists

This is called a **Tumbling Window**.

---

# 2. Problem with Tumbling Windows

Now imagine this requirement.

Suppose you are monitoring a patient’s body temperature using sensors.

You want:

> “Maximum temperature observed in the MOST RECENT 15 minutes.”

But management also says:

> “Refresh this information every 5 minutes.”

Now think carefully.

If you use tumbling windows:

* First window → 9:40 to 9:55
* Second window → 9:55 to 10:10

But this does NOT represent:

> “Most recent 15 minutes every 5 minutes.”

Because tumbling windows jump completely.

We need windows that:

* Keep the same size
* But move gradually

This introduces:

# Sliding Windows

---

# 3. What is a Sliding Window?

A sliding window:

* Has a fixed size
* But windows OVERLAP
* Windows move by a smaller interval called:

# Slide Interval

---

# 4. Example of Sliding Window

Suppose:

* Window Size = 15 minutes
* Slide Interval = 5 minutes

Then windows become:

| Window Start | Window End |
| ------------ | ---------- |
| 9:40         | 9:55       |
| 9:45         | 10:00      |
| 9:50         | 10:05      |
| 9:55         | 10:10      |
| 10:00        | 10:15      |

Notice carefully.

Each new window:

* Still has 15-minute duration
* But moves only by 5 minutes

This means windows overlap.

For example:

Window 1:
9:40 → 9:55

Window 2:
9:45 → 10:00

Overlap:
9:45 → 9:55

This overlap is the key idea behind sliding windows.

---

# 5. Real-Life Meaning

Suppose you are monitoring temperature.

At 10:00 AM, you want:

> “What was the maximum temperature observed between 9:45 and 10:00?”

Then at 10:05:

> “What was the maximum temperature observed between 9:50 and 10:05?”

Then at 10:10:

> “What was the maximum temperature observed between 9:55 and 10:10?”

This is called:

# Moving Aggregation

Because the window continuously moves forward.

---

# 6. Input Dataset

Suppose Kafka receives sensor readings like this:

```json
{
  "created_time": "2024-01-01 09:47:00",
  "reading": 36.5
}
```

Fields:

| Field        | Meaning        |
| ------------ | -------------- |
| created_time | Event Time     |
| reading      | Sensor Reading |

Kafka raw table structure:

| key       | value         |
| --------- | ------------- |
| sensor_id | complete JSON |

---

# 7. Expected Output

We want output like this:

| Sensor ID | Window Start | Window End | Max Reading |
| --------- | ------------ | ---------- | ----------- |
| S1        | 09:40        | 09:55      | 36.2        |
| S1        | 09:45        | 10:00      | 36.5        |
| S1        | 09:50        | 10:05      | 36.8        |

Meaning:

For every sliding 15-minute window:

* Calculate maximum reading
* Produce result every 5 minutes

---

# 8. Important Observation

In tumbling windows:

Each event belongs to ONE window only.

But in sliding windows:

# One event can belong to MULTIPLE windows.

Example:

Event at:
10:02

Can belong to:

* 9:50 → 10:05
* 9:55 → 10:10
* 10:00 → 10:15

This is because windows overlap.

This is one of the biggest differences between:

| Tumbling Window             | Sliding Window                |
| --------------------------- | ----------------------------- |
| Non-overlapping             | Overlapping                   |
| Event belongs to one window | Event belongs to many windows |

---

# 9. Spark Structured Streaming Implementation

Now let us implement this in Spark.

---

# 10. Step 1 — Define Schema

We first define schema for incoming JSON.

```python
StructType([
    StructField("created_time", StringType()),
    StructField("reading", DoubleType())
])
```

---

# 11. Step 2 — Read Kafka Table

```python
spark.readStream.table("kafka_bz")
```

This gives raw streaming dataframe.

---

# 12. Step 3 — Parse JSON

We use:

```python
from_json()
```

to convert JSON string into proper structure.

---

# 13. Step 4 — Convert Event Time

Very important.

Window functions require:

# Timestamp Column

So we convert:

```python
created_time
```

from string into timestamp.

```python
to_timestamp()
```

---

# 14. Step 5 — Apply Watermark

Before grouping:

```python
.withWatermark("created_time", "30 minutes")
```

Meaning:

* Spark will wait 30 minutes for late events
* After that state can be cleaned

This prevents unlimited state store growth.

---

# 15. Step 6 — Create Sliding Window

This is the most important step.

We use:

```python
window(
    "created_time",
    "15 minutes",
    "5 minutes"
)
```

Notice carefully.

Window function now has:

| Parameter    | Meaning           |
| ------------ | ----------------- |
| created_time | Event Time Column |
| 15 minutes   | Window Size       |
| 5 minutes    | Slide Interval    |

This THIRD parameter creates sliding windows.

---

# 16. Tumbling vs Sliding Syntax

## Tumbling Window

```python
window("created_time", "15 minutes")
```

Only window size.

---

## Sliding Window

```python
window(
    "created_time",
    "15 minutes",
    "5 minutes"
)
```

Window size + slide interval.

---

# 17. Aggregation

We group by:

* Sensor ID
* Sliding Window

Then calculate:

```python
max("reading")
```

Example:

```python
.groupBy(
    "sensor_id",
    window("created_time", "15 minutes", "5 minutes")
)
.agg(max("reading"))
```

---

# 18. Final Output Columns

Window column is a struct.

It contains:

* start
* end

So we extract:

```python
window.start
window.end
```

Final output:

| sensor_id | start | end | max_reading |
| --------- | ----- | --- | ----------- |

---

# 19. Watermark Still Applies

Everything we learned earlier still applies here.

Spark:

* Maintains state store
* Handles late events
* Recalculates aggregates if needed
* Cleans old windows after watermark expiry

So sliding windows also support:

# Late Event Handling

and

# State Store Cleanup

---

# 20. Important Internal Behavior

Because windows overlap:

Spark must maintain:

* More windows
* More states
* More aggregation groups

Compared to tumbling windows.

Therefore:

# Sliding windows are more expensive.

They require:

* More memory
* More state management
* More processing

But they provide:

# Better real-time moving analytics

---

# 21. Testing Strategy

For testing:

We prepared:

* 8 input records
* Expected CSV output

Then:

1. Insert input events
2. Run streaming query
3. Read actual output
4. Read expected CSV
5. Compare both datasets

This is an excellent testing strategy for streaming systems.

---

# 22. Key Learning Summary

# What is Sliding Window?

A fixed-size window that continuously moves by a smaller interval.

---

# Key Properties

| Property                  | Sliding Window |
| ------------------------- | -------------- |
| Fixed Size                | Yes            |
| Overlapping               | Yes            |
| Moving Aggregates         | Yes            |
| Event in Multiple Windows | Yes            |

---

# Syntax

```python
window(
    timestamp_column,
    window_size,
    slide_interval
)
```

---

# Example

```python
window(
    "created_time",
    "15 minutes",
    "5 minutes"
)
```

Meaning:

* 15-minute window
* Sliding every 5 minutes

---

# Watermark

```python
.withWatermark("created_time", "30 minutes")
```

Used for:

* Late event handling
* State cleanup

---

# Common Use Cases

Sliding windows are heavily used in:

* Patient monitoring systems
* CPU monitoring
* Fraud detection
* Sensor analytics
* Website traffic analysis
* Stock market analytics
* IoT streaming systems
* Real-time alerting systems

---

# Final Mental Model

Think like this:

## Tumbling Window

“Give me summaries in separate boxes.”

---

## Sliding Window

“Continuously show me the latest moving summary.”

---

# Final Conclusion

Sliding windows are one of the most powerful tools in streaming analytics.

They allow us to compute:

* Moving averages
* Moving maximums
* Rolling totals
* Rolling trends
* Near real-time analytics

The implementation in Spark is almost identical to tumbling windows.

The only major difference is:

# Adding the slide interval.

```python
window(
    event_time,
    window_size,
    slide_interval
)
```

That single extra parameter transforms:

* Tumbling Window
  into
* Sliding Window

And unlocks powerful real-time analytics capabilities.

That’s all for this lecture.

Keep learning and keep growing.

